In [7]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

BASE_PATH = Path(".")
TRAIN_PATH = BASE_PATH / "train.csv"
TEST_PATH = BASE_PATH / "test.csv"

ROW_ID = "row_id"
TARGET = "monthly_consumption_kwh"

In [8]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 71 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   household_id               120000 non-null  object 
 1   state                      120000 non-null  object 
 2   location                   120000 non-null  object 
 3   climate_region             120000 non-null  object 
 4   dwelling_type              120000 non-null  object 
 5   floor_number               119310 non-null  float64
 6   building_age               120000 non-null  int64  
 7   floor_area_sqft            119465 non-null  float64
 8   airconditioned_area_pct    120000 non-null  int64  
 9   num_rooms                  120000 non-null  int64  
 10  num_bedrooms               119262 non-null  float64
 11  num_bathrooms              120000 non-null  int64  
 12  wall_material              120000 non-null  object 
 13  roof_material              12

In [9]:
cols_to_drop = ["floor_number", "building_age", "floor_area_sqft", "wall_material", "num_children", "num_elderly", "year", "month", "num_holidays", "working_days", "weekend_days", "outage_hours",  'air_temp_min_c',
       'air_temp_max_c', 'air_temp_mean_c', 'air_humidity_min_pct',
       'air_humidity_max_pct', 'air_humidity_mean_pct', 'rainfall_min_mm_day',
       'rainfall_max_mm_day', 'rainfall_mean_mm_day', 'rainfall_total_mm',
       'solar_irradiance_min_wm2', 'solar_irradiance_max_wm2',
       'solar_irradiance_mean_wm2', 'wind_speed_min_mps', 'wind_speed_max_mps',
       'wind_speed_mean_mps', 'monthly_consumption_kwh']

train_red = train_df.drop([], axis=1)

numeric_cols = train_red.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in train_red.columns if c not in numeric_cols]

for col in numeric_cols:
       train_red[col] = train_red[col].fillna(train_red[col].mean())

for col in cat_cols:
       train_red[col] = train_red[col].fillna(train_red[col].mode())
       
print("Numeric Features : ", len(numeric_cols))
print("Category Columns : ", len(cat_cols))


Numeric Features :  58
Category Columns :  13


In [10]:
X = train_red.drop([TARGET, ROW_ID], axis=1, errors="ignore")
y = train_red[TARGET].astype(float)

X_test = test_df.copy()
row_id_test = X_test[ROW_ID].values

X = pd.get_dummies(X, drop_first=True)

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)



In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor()
model.fit(X_train, y_train)